In [6]:
import os
import sys

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
from datasets import Dataset

from utils.helpers import get_system_prompt, get_task_type, get_default_response, make_prompt, parse_output, make_demonstrations, get_allowed_concepts

from utils.plots import GT_COLS

from utils.constants import PIPE_MAX_NEW_TOKENS, MODEL_TEMPERATURE, BATCH_SIZE, PIPE_RETURN_FULL_TEXT, CONCEPT_TO_CHAPTER_MAPPING

In [18]:
data = "./data/final_dataset.csv"

df = pd.read_csv(data, sep=";")
df = df.iloc[::-1]

In [19]:
demo_labels = [
    ["no", "no", "yes"], # All wrong
    ["yes", "no", "no"], # Correct theme, wrong topic
    ["yes", "yes", "no"], # All correct 
    ["yes", "yes", "yes"], # Extra concept
    ["no", "no", "no"], # Mismatch of theme, topic
    ["yes", "no", "yes"] # Topic, concept wrong
]

label_cols = df[GT_COLS]

In [20]:
indices = []

for labels in demo_labels:
    match = (label_cols == labels).all(axis=1)
    idx = match.idxmax() if match.any() else None
    indices.append(idx)

dataset = Dataset.from_pandas(df.loc[indices])
demos = make_demonstrations(dataset)
indices

[534, 60, 272, 253, 114, 446]

In [21]:
print(demos)

Demonstration:

Theme: handicrafts
Topic: pottery
Allowed concepts: user input, program output, variables, arithmetics, conditional statements, logical operators

--- PROBLEM DESCRIPTION ---
In the world of The Legend of Zelda, Link has discovered various treasures in a dungeon, each rated on a scale from 1 to 5, where each number corresponds to a specific treasure quality:
<table>
<tr>
<th>Rating</th>
<th>Description</th>
</tr>
<tr>
<th>5</th>
<th>Legendary</th>
</tr>
<tr>
<th>4</th>
<th>Rare</th>
</tr>
<tr>
<th>3</th>
<th>Uncommon</th>
</tr>
<tr>
<th>2</th>
<th>Common</th>
</tr>
<tr>
<th>1</th>
<th>Junk</th>
</tr>
</table>
Write a program that asks the user for a number and prints the treasure quality related to that number. If the user enters any other number, the program should print the message <code>Invalid rating!</code>.

Below is an example of the expected operation of the program.

<pre>
What rating?
<b>&lt; 3</b>
Uncommon
</pre>

Another example.

<pre>
What rating?
<b>&lt; 